# CSV to JSON Converter

This notebook converts CSV files to JSON format. It supports all CSV files in the workspace and provides options for different JSON orientations.

## 1. Import Required Libraries

In [109]:
import pandas as pd
import json
import os
from pathlib import Path

# Install fastavro if not already present
try:
    import fastavro
    from fastavro import writer, parse_schema
    print(f"fastavro {fastavro.__version__} ready")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fastavro"])
    import fastavro
    from fastavro import writer, parse_schema
    print(f"fastavro {fastavro.__version__} installed and ready")

fastavro 1.12.1 ready


## 2. Load CSV File

Set the path to the CSV file you want to convert. All available CSV files in the workspace are listed below.

In [110]:
# Base directory for the workspace
BASE_DIR = Path("/Users/mac/Documents/GitHub/DS-IOH-Application-Mapping")

# List all available CSV files in the workspace
csv_files = list(BASE_DIR.glob("*.csv"))
print("Available CSV files:")
for i, f in enumerate(csv_files):
    print(f"  [{i}] {f.name}")

# -------------------------------------------------------
# Set the CSV file to convert (change index or provide path)
# -------------------------------------------------------
CSV_FILE = csv_files[1]   # <-- change index to select a different file
# Or specify directly:
# CSV_FILE = BASE_DIR / "rnr_app_category_v2.csv"

print(f"\nSelected: {CSV_FILE.name}")

# Load the CSV into a DataFrame
df = pd.read_csv(CSV_FILE, encoding="utf-8")
print(f"Loaded {len(df)} rows × {len(df.columns)} columns")

Available CSV files:
  [0] Signature Apps Library 20250828(Tracker v4 20240910).csv
  [1] rnr_app_category_v2.csv
  [2] taxonomy_reference.csv
  [3] mis_app_category.csv
  [4] mis_app_category_v2.csv

Selected: rnr_app_category_v2.csv
Loaded 1199 rows × 8 columns


## 3. Explore CSV Data

Preview the data to understand its structure before converting.

In [111]:
# First few rows
print("=== First 5 rows ===")
display(df.head())

# Column names and data types
print("\n=== Column info ===")
print(df.dtypes)

# Shape and nulls
print(f"\nShape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()}")

=== First 5 rows ===


,app_name,source_app_names_old,sig_app_tags,category,subcategory,secondary_category,secondary_subcategory,description
0,1Cak,[1cak],1cak,entertainment,comedy_memes,NaN,NaN,1Cak is a comedy & memes app by PT 1Cak Media....
1,2DFire,[2DFire],2DFire,productivity_tools,business_operations,NaN,NaN,2DFire is a business operations app by 2DFire....
2,360Kredi,[360KREDIT],360KREDIT,finance,payment_gateway,NaN,NaN,360Kredi is a payment gateway app by PT 360 Kr...
3,4shared,[4shared],4shared,productivity_tools,cloud_storage_file_sharing,NaN,NaN,4shared is a cloud storage app by 4shared. It ...
4,7-Eleven,[7_Eleven],7_Eleven,commerce,grocery,NaN,NaN,7-Eleven is a grocery app by Seven & i Holding...



=== Column info ===
app_name                 str
source_app_names_old     str
sig_app_tags             str
category                 str
subcategory              str
secondary_category       str
secondary_subcategory    str
description              str
dtype: object

Shape: (1199, 8)

Null counts:
app_name                    0
source_app_names_old        0
sig_app_tags                0
category                    0
subcategory                 0
secondary_category       1157
secondary_subcategory    1157
description                 0
dtype: int64


In [112]:
from ast import literal_eval

# Robustly parse string-encoded arrays to real Python lists.
# literal_eval handles quoted strings inside lists correctly,
# e.g. "['app one, v2', 'app two']" → ['app one, v2', 'app two']
def parse_array_field(val):
    if not isinstance(val, str):
        return val
    val = val.strip()
    if val.startswith("["):
        try:
            return literal_eval(val)   # safest: respects quotes & escapes
        except Exception:
            # fallback: strip brackets and split (only for simple cases)
            return [v.strip().strip("'\"") for v in val.strip("[]").split(",") if v.strip()]
    return val

# Apply to all columns that contain list-like strings
array_columns = ["source_app_names_old", "sig_app_tags"]   # <-- add more column names here if needed
for col in array_columns:
    if col in df.columns:
        df[col] = df[col].apply(parse_array_field)

print("Array columns parsed:", array_columns)
print("Sample:", df["source_app_names_old"].iloc[0] if "source_app_names_old" in df.columns else "N/A")

Array columns parsed: ['source_app_names_old', 'sig_app_tags']
Sample: ['1cak']


## 4. Convert CSV to Avro

Output format: **Avro** — BigQuery's most reliable format for structured data.

- Arrays are preserved as Avro `array` fields → BigQuery `REPEATED` columns.
- Schema is inferred automatically from the DataFrame dtypes.
- All nullable fields use Avro union `["null", <type>]` so missing values are safe.

In [113]:
def infer_avro_type(series, array_cols):
    """Map a pandas Series dtype to a nullable Avro type."""
    if series.name in array_cols:
        # Array of nullable strings → BigQuery REPEATED STRING
        return ["null", {"type": "array", "items": ["null", "string"]}]
    dtype = series.dtype
    if dtype == "int64":
        return ["null", "long"]
    elif dtype == "float64":
        return ["null", "double"]
    elif dtype == "bool":
        return ["null", "boolean"]
    else:
        return ["null", "string"]


def build_avro_schema(df, array_cols, record_name="Row"):
    """Build and parse an Avro schema from a DataFrame."""
    fields = [
        {"name": col, "type": infer_avro_type(df[col], array_cols), "default": None}
        for col in df.columns
    ]
    return parse_schema({"type": "record", "name": record_name, "fields": fields})


def df_to_avro_records(df, array_cols):
    """Convert DataFrame rows to Avro-compatible dicts, handling NaN and lists."""
    result = []
    for row in df.to_dict(orient="records"):
        record = {}
        for k, v in row.items():
            if isinstance(v, list):
                # Stringify each element; keep None as null
                record[k] = [str(item).strip() if item is not None else None for item in v]
            elif v is None:
                record[k] = None
            elif not isinstance(v, (bool, int, float, str)) or (isinstance(v, float) and pd.isna(v)):
                record[k] = None
            else:
                record[k] = v
        result.append(record)
    return result

# convert column sig_app_tags to array of strings, previously string separated by pipes |
def parse_pipe_separated(val):
    if isinstance(val, str) and "|" in val:
        return [v.strip() for v in val.split("|") if v.strip()]
    return [val]


# Build schema and records from the loaded (and array-parsed) DataFrame
df["sig_app_tags"] = df["sig_app_tags"].apply(parse_pipe_separated)
avro_schema  = build_avro_schema(df, array_columns)
avro_records = df_to_avro_records(df, array_columns)

print(f"Schema fields : {[f['name'] for f in avro_schema['fields']]}")
print(f"Total records : {len(avro_records)}")
print(f"\nSample record :\n{json.dumps(avro_records[0], indent=2, ensure_ascii=False)}")

Schema fields : ['app_name', 'source_app_names_old', 'sig_app_tags', 'category', 'subcategory', 'secondary_category', 'secondary_subcategory', 'description']
Total records : 1199

Sample record :
{
  "app_name": "1Cak",
  "source_app_names_old": [
    "1cak"
  ],
  "sig_app_tags": [
    "1cak"
  ],
  "category": "entertainment",
  "subcategory": "comedy_memes",
  "secondary_category": null,
  "secondary_subcategory": null,
  "description": "1Cak is a comedy & memes app by PT 1Cak Media. It is Indonesia's meme and humor sharing community, inspired by 9GAG."
}


## 5. Save Avro Output

The output `.avro` file is saved to the `output/` folder.  
Load it into BigQuery with source format **AVRO** — no manual schema definition needed.

In [115]:
# Create output directory if it doesn't exist
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / (CSV_FILE.stem + ".avro")

with open(output_path, "wb") as f:
    writer(f, avro_schema, avro_records)

print(f"Saved : {output_path}")
print(f"Size  : {output_path.stat().st_size / 1024:.1f} KB")

# Quick read-back validation
with open(output_path, "rb") as f:
    sample = list(fastavro.reader(f))
print(f"\nRead-back OK — {len(sample)} records")
print("First record:", json.dumps(sample[0], indent=2, ensure_ascii=False))

Saved : /Users/mac/Documents/GitHub/DS-IOH-Application-Mapping/output/rnr_app_category_v2.avro
Size  : 247.5 KB

Read-back OK — 1199 records
First record: {
  "app_name": "1Cak",
  "source_app_names_old": [
    "1cak"
  ],
  "sig_app_tags": [
    "1cak"
  ],
  "category": "entertainment",
  "subcategory": "comedy_memes",
  "secondary_category": null,
  "secondary_subcategory": null,
  "description": "1Cak is a comedy & memes app by PT 1Cak Media. It is Indonesia's meme and humor sharing community, inspired by 9GAG."
}


### Batch Convert — All CSV Files to Avro

Run the cell below to convert **every** CSV file in the workspace to Avro at once.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

# Columns that should be treated as arrays across all files
BATCH_ARRAY_COLS = ["source_app_names_old"]  # <-- extend as needed

results = []
for csv_path in sorted(BASE_DIR.glob("*.csv")):
    try:
        batch_df = pd.read_csv(csv_path, encoding="utf-8")

        # Parse array columns
        for col in BATCH_ARRAY_COLS:
            if col in batch_df.columns:
                batch_df[col] = batch_df[col].apply(parse_array_field)

        schema  = build_avro_schema(batch_df, BATCH_ARRAY_COLS, record_name=csv_path.stem.replace(" ", "_"))
        records = df_to_avro_records(batch_df, BATCH_ARRAY_COLS)

        out_path = OUTPUT_DIR / f"{csv_path.stem}.avro"
        with open(out_path, "wb") as f:
            writer(f, schema, records)

        results.append({"file": csv_path.name, "rows": len(batch_df), "output": out_path.name, "status": "OK"})
    except Exception as e:
        results.append({"file": csv_path.name, "rows": None, "output": None, "status": str(e)})

summary_df = pd.DataFrame(results)
display(summary_df)